 # Introduction to local search methods

 This notebook provides a simple implementation of **hill-clibing** and **local beam** search methods.

 **Important note:**

The Python code was created with the help of a generative AI assistant, so some fragments were drafted based on code taken from web.

### Common functions definition

In [1]:
# Imports
import random

# Checks if it is safe to place a queen at the given position.
def is_safe(board, row, col):
  for i in range(row):
    if board[i] == col or \
       abs(board[i] - col) == row - i:
      return False
  return True

# Calculates the number of conflicts in the current board.
def evaluate(board):
  conflicts = 0
  for i in range(len(board)):
    for j in range(i + 1, len(board)):
      if board[i] == board[j] or \
         abs(board[i] - board[j]) == j - i:
        conflicts += 1
  return conflicts

### Hill-clibing search

In [2]:
# Source: https://github.com/Mayur9167/AI_Puzzle_Solution

def hill_climbing(board_size):
  """Performs the hill climbing algorithm for the n-queens problem."""
  current_board = [random.randint(0, board_size - 1) for _ in range(board_size)]
  current_cost = evaluate(current_board)
  # print(current_board)

  while True:
    neighbors = []
    for i in range(board_size):
      for j in range(board_size):
        if current_board[i] != j:
          neighbor_board = list(current_board)
          neighbor_board[i] = j
          neighbors.append(neighbor_board)

    best_neighbor = None
    best_neighbor_cost = float('inf')

    for neighbor in neighbors:
      neighbor_cost = evaluate(neighbor)
      if neighbor_cost < best_neighbor_cost:
        best_neighbor_cost = neighbor_cost
        best_neighbor = neighbor

    if best_neighbor is None or best_neighbor_cost >= current_cost:
      return current_board, current_cost

    current_board = best_neighbor
    current_cost = best_neighbor_cost

### Local beam search


In [3]:
# Source:  https://github.com/Mayur9167/AI_Puzzle_Solution

def local_beam_search(board_size, beam_width):
  """Performs the local beam search algorithm for the n-queens problem."""
  current_boards = [[random.randint(0, board_size - 1) for _ in range(board_size)] for _ in range(beam_width)]
  current_costs = [evaluate(board) for board in current_boards]

  while True:
    neighbors = []
    for board in current_boards:
      for i in range(board_size):
        for j in range(board_size):
          if board[i] != j:
            neighbor_board = list(board)
            neighbor_board[i] = j
            neighbors.append(neighbor_board)

    neighbor_costs = [evaluate(neighbor) for neighbor in neighbors]

    sorted_neighbors = sorted(zip(neighbors, neighbor_costs), key=lambda x: x[1])
    new_boards = [board for board, cost in sorted_neighbors[:beam_width]]
    new_costs = [cost for board, cost in sorted_neighbors[:beam_width]]

    if new_costs[0] == 0:
      return new_boards[0], new_costs[0]

    if new_costs == current_costs:
      return current_boards[0], current_costs[0]


    current_boards = new_boards
    current_costs = new_costs

### Run the algorithm

In [8]:
# Example usage for the 4-queens problem
board_size = 8
beam_width = 30

# solution, cost = hill_climbing(board_size)
solution, cost = local_beam_search(board_size, beam_width)

if cost == 0:
  print("Solution found:")
  for row in range(board_size):
    line = ""
    for col in range(board_size):
      if solution[row] == col:
        line += "Q "
      else:
        line += ". "
    print(line)
else:
  print("Local optimum found with", cost, "conflicts.")


Solution found:
Q . . . . . . . 
. . . . . Q . . 
. . . . . . . Q 
. . Q . . . . . 
. . . . . . Q . 
. . . Q . . . . 
. Q . . . . . . 
. . . . Q . . . 


---
## Exercises
1. Create a function that runs the an algorithm (e.g. hill_climbing) n times (default = 100) and counts the number of successes and failures (local otima).

2. Is local search applicable to finding a solution for the next problems?
  * 8 puzzle
  * Traveling salesman

In [ ]:
# prompt: create a function that takes an algorithm as a input (e.g. hill-clibing), runs it n times (default = 100) and returns the number of successes and failures.

def run_algorithm_multiple_times(algorithm, board_size, n=100, beam_width=None):
    """Runs the specified algorithm multiple times and counts successes/failures.

    Args:
        algorithm: The algorithm function to run (e.g., hill_climbing, local_beam_search).
        board_size: The size of the board for the n-queens problem.
        n: The number of times to run the algorithm (default: 100).
        beam_width: The beam width for local beam search (required if using that algorithm)

    Returns:
        A tuple containing the number of successes and failures.
    """
    successes = 0
    failures = 0

    for _ in range(n):
        if algorithm == hill_climbing:
            solution, cost = algorithm(board_size)
        elif algorithm == local_beam_search:
          if beam_width is None:
            raise ValueError("beam_width must be provided for local_beam_search")
          solution, cost = algorithm(board_size, beam_width)
        else:
          raise ValueError("Invalid algorithm specified")

        if cost == 0:
            successes += 1
        else:
            failures += 1

    return successes, failures

In [ ]:
# successes, failures = run_algorithm_multiple_times(local_beam_search, 8, beam_width=30)
successes, failures = run_algorithm_multiple_times(hill_climbing, 8, beam_width=30)
print(f"Successes: {successes}, Failures: {failures}")


Successes: 7, Failures: 93
